In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

# =========================
# STEP 1: Load Data
# =========================

# Load the cleaned dataset provided in the data folder
df = pd.read_csv("../data/restaurant_reviews_cleaned.csv")

# Inspect basic structure
df.head()

# =========================
# STEP 2: Feature Representation
# =========================

# Define feature text and target labels
X_text = df["Review_clean"]
y = df["Sentiment"]

print("Number of reviews:", len(X_text))
print("\nSentiment distribution:")
print(y.value_counts())

# Initialize TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    max_features=5000,
    min_df=2,
    max_df=0.95
)

# Check for invalid cleaned reviews
print("NaNs in Review_clean:", df["Review_clean"].isna().sum())
print("Empty strings in Review_clean:", (df["Review_clean"] == "").sum())

# Drop rows with NaN cleaned text
df = df[df["Review_clean"].notna()].copy()

# Redefine features and labels after dropping rows
X_text = df["Review_clean"]
y = df["Sentiment"]

print("Rows remaining after dropping invalid reviews:", len(df))

# Fit TF-IDF
X_tfidf = tfidf_vectorizer.fit_transform(X_text)

print("TF-IDF matrix shape:", X_tfidf.shape)

# =========================
# STEP 3: Modeling
# =========================

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

print("\nTraining sentiment distribution:")
print(y_train.value_counts())

# Naive Bayes Model
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

print("Naive Bayes model trained.")

# Logistic Regression Model
lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    solver="lbfgs"
)

lr_model.fit(X_train, y_train)

print("Logistic Regression model trained.")

# =========================
# STEP 4: Predictions
# =========================

nb_preds = nb_model.predict(X_test)
lr_preds = lr_model.predict(X_test)

print("Predictions generated for both models.")

# =========================
# STEP 5: Evaluation
# =========================

# Accuracy
nb_accuracy = np.mean(nb_preds == y_test)
lr_accuracy = np.mean(lr_preds == y_test)

print("Naive Bayes Accuracy:", nb_accuracy)
print("Logistic Regression Accuracy:", lr_accuracy)

# Confusion Matrices
print("Naive Bayes confusion matrix:")
ConfusionMatrixDisplay.from_predictions(y_test, nb_preds)
plt.show()

print("Logistic Regression confusion matrix:")
ConfusionMatrixDisplay.from_predictions(y_test, lr_preds)
plt.show()

# Classification Reports
print("Naive Bayes classification report")
print(classification_report(y_test, nb_preds))

print("Logistic Regression classification report")
print(classification_report(y_test, lr_preds))

# =========================
# STUDENT ANALYSIS SECTION
# =========================

# TODO: Answer the following questions:
# - Which model performs better?
# - Which sentiment class is hardest to predict?
# - Why might neutral reviews be difficult to classify?
# - What impact does class imbalance have on performance?